In [161]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
from torch_geometric.utils import from_scipy_sparse_matrix
from torch_geometric.data import Data
import scipy.sparse as sp
import pandas as pd
import networkx as nx
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d
import numpy as np
import pandas as pd
import networkx as nx
import pandas as pd
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
from torch_geometric.utils import from_scipy_sparse_matrix
from torch_geometric.data import Data
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [163]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels, num_layers=3, dropout_rate=0.01):
        super(GCNEncoder, self).__init__()
        
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * out_channels, cached=False))
        
        # Add intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * out_channels, 2 * out_channels, cached=False))
        
        # Final layer: hidden size to output size
        self.convs.append(GCNConv(2 * out_channels, out_channels, cached=False))
        
        # Batch normalization layers for each convolution layer
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * out_channels) for _ in range(num_layers - 1)])
        
        # Dropout rate
        self.dropout = torch.nn.Dropout(p=dropout_rate)
        
        # LeakyReLU instead of ReLU
        self.activation = torch.nn.LeakyReLU(negative_slope=0.05)
        #self.activation = torch.nn.ReLU()

    def forward(self, x, edge_index):
        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        x = self.activation(x)
        x = F.relu(x)
        
        # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x  # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x)  # Apply batch normalization
            x = self.activation(x)
            x = self.dropout(x)
            
            
            # Add residual connection (skip connection)
            x = x + x_residual
        
        # Final layer (no activation or batch norm)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        return x

In [165]:

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GCN2Conv, BatchNorm
from torch.nn import Linear
from torch_geometric.utils import dropout_adj

class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2, dropout_rate=0.01):
        super(GCNEncoder, self).__init__()  # Updated the class name here
        
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
        # Add intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
        # Final layer: hidden size to output size
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
        
        # Batch normalization layers for each convolution layer
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels) for _ in range(num_layers - 1)])
        
        # Dropout layer
        self.dropout = torch.nn.Dropout(p=dropout_rate)
        
        # Activation function
        self.activation = torch.nn.LeakyReLU(negative_slope=0.05)

    def forward(self, x, edge_index):
        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        x = self.activation(x)
        
        # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x  # Residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x)  # Apply batch normalization
            x = self.activation(x)
            x = self.dropout(x)
            
            # Add residual connection (skip connection)
            x = x + x_residual
        
        # Final layer (no activation or batch norm)
        x = self.convs[-1](x, edge_index)
        return x



# Normalize the adjacency matrix
def normalize_adj(adj):
    """Symmetrically normalize adjacency matrix."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    return adj.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt).tocoo()

# GAE model with weight decay and dropout
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=16, epochs=500, dropout_rate=0.01, weight_decay=1e-4):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    # Normalize the adjacency matrix
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    # Get edge_index and edge_weight
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)
    # Define model
    model = GAE(GCNEncoder(x.shape[1], embedding_dim, dropout_rate=dropout_rate))
    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)  # Applying weight decay le = 0.05
    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()
    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
    return adj_reconstructed

# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges


In [167]:
# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.1filter.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

# Step 1: Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")


Loaded data with shape: (100, 1700)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 100
Node features shape: (100, 100)
Ground truth loaded with shape: (100, 100)
Number of known edges: 175


In [169]:
# Class definitions (GCNEncoder, normalize_adj, apply_gae_to_adjacency, etc.) here

# Define the function to perform the GAE process for each threshold and multiple iterations
def run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations=3):
    num_genes = data.shape[0]
    node_features = np.identity(num_genes)
    all_results = {}
    iteration_sums = {}

    for threshold in thresholds:
        print(f"Running for threshold: {threshold}")
        
        # Create a dictionary to store the results for each algorithm and threshold
        all_results[f'threshold_{threshold}'] = {}
        iteration_sums[f'threshold_{threshold}'] = {}

        # Initialize the iteration sum matrices for each threshold
        for iter_num in range(num_iterations):
            iteration_sums[f'threshold_{threshold}'][f'iter{iter_num + 1}_GCN_sum'] = np.zeros((num_genes, num_genes))

        for alg in algorithms:
            print(f"Running GAE for algorithm: {alg} with threshold: {threshold}")
            alg_columns = [col for col in data.columns if col.startswith(alg + '.')]
            alg_matrix = data[alg_columns].values

            # For each algorithm, create a list to store multiple runs
            all_results[f'threshold_{threshold}'][alg] = []

            for iteration in range(num_iterations):
                print(f"Iteration {iteration + 1}/{num_iterations} for algorithm: {alg}")
                
                # Apply GAE to the adjacency matrix of the algorithm
                adj_reconstructed = apply_gae_to_adjacency(alg_matrix, node_features)
                
                # Binarize the reconstructed adjacency matrix using the current threshold
                adj_binary = (adj_reconstructed > threshold).astype(int)

                # Store the binary adjacency matrix for this run
                all_results[f'threshold_{threshold}'][alg].append(adj_binary)

                # Add the binary matrix to the sum for this iteration and threshold
                iteration_sums[f'threshold_{threshold}'][f'iter{iteration + 1}_GCN_sum'] += adj_binary

    return all_results, iteration_sums

In [171]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
import scipy.sparse as sp
from torch_geometric.nn import GAE
from torch.nn import BatchNorm1d



def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=64, epochs=200, dropout_rate=0.01, weight_decay=5e-4):
    """
    Applies Graph Autoencoder (GAE) to an adjacency matrix.
    
    Parameters:
    adj_matrix (numpy.ndarray): The adjacency matrix.
    node_features (numpy.ndarray): Node feature matrix.
    embedding_dim (int): Embedding dimensions for the encoder.
    epochs (int): Number of training epochs.
    dropout_rate (float): Dropout rate.
    weight_decay (float): Weight decay for optimizer.

    Returns:
    numpy.ndarray: Reconstructed adjacency matrix.
    """
    # Prepare graph data for PyTorch Geometric
    edge_index = from_scipy_sparse_matrix(sp.csr_matrix(adj_matrix))[0]
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define model (Adjust out_channels as necessary)
    hidden_channels = embedding_dim
    out_channels = embedding_dim  # Set this to the required dimension for the output embeddings
    model = GAE(GCNEncoder(x.shape[1], hidden_channels, out_channels, dropout_rate=dropout_rate))

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, data = model.to(device), data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)

    # Train the GAE model
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)  # Encoding step
        loss = model.recon_loss(z, data.edge_index)  # Reconstruction loss
        loss.backward()
        optimizer.step()

    # Evaluate the model
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()

    return adj_reconstructed


# Function to apply a second binarization threshold to the summed adjacency matrices
def apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds):
    binary_sums = {}
    
    for threshold in thresholds:
        binary_sums[f'threshold_{threshold}'] = {}
        for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
            binary_sums[f'threshold_{threshold}'][iter_num] = {}

            for iter_threshold in iter_thresholds:
                print(f"Applying iter_threshold {iter_threshold} to {iter_num} for threshold {threshold}")
                
                # Apply binarization on the summed adjacency matrix
                summed_matrix = iteration_sums[f'threshold_{threshold}'][iter_num]
                binary_matrix = (summed_matrix >= iter_threshold).astype(int)

                # Store the result
                binary_sums[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}'] = binary_matrix
    
    return binary_sums

# Define thresholds and number of iterations
#thresholds = [0, 0.2,0.4,0.6, 0.8]  # Different thresholds to run
thresholds = [0.6]  # Different thresholds to run
num_iterations = 20  # Number of runs for each threshold

# Run the GAE process and store results for each threshold and algorithm
aggregated_results, iteration_sums = run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations)

# Define the additional binarization thresholds for the summed matrices
#iter_thresholds = [2,4,6, 8,10,12,14,16]  # Example thresholds for binarizing summed matrices
iter_thresholds = [8,10]  # Example thresholds for binarizing summed matrices


# Apply the iteration-level thresholding
binary_summed_matrices = apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds)

# Example: Access binarized summed matrices for a specific iteration and iter_threshold
for threshold in thresholds:
    for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
        for iter_threshold in iter_thresholds:
            binarized_matrix = binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']
            print(f"Binarized adjacency matrix for {iter_num}, threshold {threshold}, iter_threshold {iter_threshold}:")
            print(binarized_matrix)

            # You can store or analyze these matrices further
            output_file = f'binarized_{iter_num}_{threshold}_iter_thresh_{iter_threshold}.csv'
            #pd.DataFrame(binarized_matrix).to_csv(output_file)





Running for threshold: 0.6
Running GAE for algorithm: GRN_ARACNE with threshold: 0.6
Iteration 1/20 for algorithm: GRN_ARACNE
Iteration 2/20 for algorithm: GRN_ARACNE
Iteration 3/20 for algorithm: GRN_ARACNE
Iteration 4/20 for algorithm: GRN_ARACNE
Iteration 5/20 for algorithm: GRN_ARACNE
Iteration 6/20 for algorithm: GRN_ARACNE
Iteration 7/20 for algorithm: GRN_ARACNE
Iteration 8/20 for algorithm: GRN_ARACNE
Iteration 9/20 for algorithm: GRN_ARACNE
Iteration 10/20 for algorithm: GRN_ARACNE
Iteration 11/20 for algorithm: GRN_ARACNE
Iteration 12/20 for algorithm: GRN_ARACNE
Iteration 13/20 for algorithm: GRN_ARACNE
Iteration 14/20 for algorithm: GRN_ARACNE
Iteration 15/20 for algorithm: GRN_ARACNE
Iteration 16/20 for algorithm: GRN_ARACNE
Iteration 17/20 for algorithm: GRN_ARACNE
Iteration 18/20 for algorithm: GRN_ARACNE
Iteration 19/20 for algorithm: GRN_ARACNE
Iteration 20/20 for algorithm: GRN_ARACNE
Running GAE for algorithm: GRN_BC3NET with threshold: 0.6
Iteration 1/20 for algorit

In [173]:
import numpy as np
import pandas as pd

# Function to sum across iterations and apply a common threshold
def sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold):
    common_sums = {}

    for threshold in thresholds:
        common_sums[f'threshold_{threshold}'] = {}
        for iter_threshold in iter_thresholds:
            print(f"Summing across iterations for threshold {threshold} and iter_threshold {iter_threshold}")

            # Initialize the matrix for summing across iterations
            iteration_sum_matrix = np.zeros_like(binary_summed_matrices[f'threshold_{threshold}']['iter1_GCN_sum'][f'iter_threshold_{iter_threshold}'])

            # Sum the matrices across all iterations
            for iter_num in binary_summed_matrices[f'threshold_{threshold}'].keys():
                iteration_sum_matrix += binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']

            # Apply the common threshold: if sum >= common_threshold, set 1; else set 0
            final_binarized_matrix = (iteration_sum_matrix >= common_threshold).astype(int)

            # Store the result
            common_sums[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}'] = final_binarized_matrix

    return common_sums

# Load the binary summed matrices from previous steps (from apply_iteration_threshold function)
# Example: binary_summed_matrices = your result from the previous step

# Define the common threshold
common_threshold = 15  # This is the threshold where if sum >= 30 across iterations, we set the value to 1. Consistently found in many iterations

# Sum across iterations and apply the common threshold
final_common_binarized_matrices = sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold)

# Example: Access and print the final binarized matrix for a specific threshold and iteration threshold
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        print(f"Final binarized matrix for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        print(final_matrix)

        # You can save these matrices to CSV or further analyze them
        output_file = f'final_binarized_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.csv'
        #pd.DataFrame(final_matrix).to_csv(output_file)


Summing across iterations for threshold 0.6 and iter_threshold 8
Summing across iterations for threshold 0.6 and iter_threshold 10
Final binarized matrix for threshold 0.6, iter_threshold 8, common_threshold 15:
[[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 1]
 [0 0 0 ... 0 1 1]]
Final binarized matrix for threshold 0.6, iter_threshold 10, common_threshold 15:
[[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 1]
 [0 0 0 ... 0 1 1]]


In [175]:
import pandas as pd
import numpy as np

# Assuming `final_common_binarized_matrices` contains your final matrices

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges

# Assuming the gene names are the row/column names from your original data
gene_names = list(data.index)  # Replace 'data' with your actual DataFrame that contains the gene names

# Example: Use the final binarized matrix from the previous step (e.g., for threshold 0.8, iter_threshold 6, common_threshold 2)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (edges)
        edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Print the edges
        print(f"Gene interactions for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        for edge in edges:
            print(edge)

        # Optionally, save the edges to a file
        output_file = f'gene_interactions_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.txt'
        with open(output_file, 'w') as f:
            for edge in edges:
                f.write(edge + '\n')


Gene interactions for threshold 0.6, iter_threshold 8, common_threshold 15:
G1-G16
G1-G23
G1-G24
G1-G25
G1-G29
G1-G36
G1-G37
G1-G48
G1-G61
G1-G68
G1-G69
G1-G72
G1-G73
G1-G74
G1-G76
G1-G77
G1-G89
G10-G12
G10-G14
G10-G16
G10-G17
G10-G19
G10-G20
G10-G31
G10-G38
G10-G39
G10-G43
G10-G44
G10-G45
G10-G5
G10-G59
G10-G63
G10-G64
G10-G78
G10-G79
G10-G80
G10-G9
G10-G91
G10-G92
G10-G93
G10-G94
G10-G95
G100-G39
G100-G83
G100-G85
G100-G9
G100-G90
G100-G93
G11-G14
G11-G15
G11-G17
G11-G2
G11-G21
G11-G26
G11-G27
G11-G30
G11-G32
G11-G34
G11-G36
G11-G38
G11-G5
G11-G54
G11-G56
G11-G6
G11-G8
G11-G81
G11-G82
G11-G84
G11-G86
G11-G98
G11-G99
G12-G15
G12-G17
G12-G19
G12-G20
G12-G38
G12-G39
G12-G42
G12-G5
G12-G6
G12-G83
G12-G86
G12-G87
G12-G91
G12-G92
G12-G98
G14-G15
G14-G16
G14-G17
G14-G38
G14-G39
G14-G43
G14-G44
G14-G45
G14-G56
G14-G63
G14-G64
G14-G9
G14-G91
G14-G92
G15-G17
G15-G2
G15-G21
G15-G26
G15-G27
G15-G30
G15-G32
G15-G34
G15-G36
G15-G42
G15-G5
G15-G54
G15-G6
G15-G60
G15-G8
G15-G81
G15-G82
G15-G84
G15-G

In [177]:
edges

['G1-G25',
 'G1-G29',
 'G1-G37',
 'G1-G48',
 'G1-G61',
 'G1-G68',
 'G1-G69',
 'G1-G72',
 'G1-G73',
 'G1-G74',
 'G1-G77',
 'G10-G12',
 'G10-G14',
 'G10-G16',
 'G10-G17',
 'G10-G39',
 'G10-G43',
 'G10-G44',
 'G10-G45',
 'G10-G63',
 'G10-G64',
 'G10-G79',
 'G10-G9',
 'G10-G91',
 'G10-G92',
 'G10-G93',
 'G100-G39',
 'G100-G9',
 'G11-G15',
 'G11-G17',
 'G11-G26',
 'G11-G27',
 'G11-G5',
 'G11-G6',
 'G11-G98',
 'G11-G99',
 'G12-G5',
 'G12-G86',
 'G12-G98',
 'G14-G16',
 'G14-G17',
 'G14-G38',
 'G14-G43',
 'G14-G44',
 'G14-G63',
 'G14-G91',
 'G14-G92',
 'G15-G17',
 'G15-G21',
 'G15-G26',
 'G15-G27',
 'G15-G30',
 'G15-G34',
 'G15-G5',
 'G15-G54',
 'G15-G6',
 'G15-G81',
 'G15-G82',
 'G15-G98',
 'G15-G99',
 'G16-G17',
 'G16-G52',
 'G16-G56',
 'G17-G38',
 'G17-G43',
 'G17-G56',
 'G17-G91',
 'G17-G92',
 'G18-G22',
 'G18-G37',
 'G18-G46',
 'G18-G47',
 'G18-G48',
 'G18-G52',
 'G18-G53',
 'G18-G56',
 'G18-G57',
 'G18-G58',
 'G18-G60',
 'G19-G20',
 'G19-G28',
 'G19-G38',
 'G19-G44',
 'G19-G45',
 'G19-G5

In [179]:
data

,GRN_ARACNE.G1,GRN_ARACNE.G10,GRN_ARACNE.G100,GRN_ARACNE.G11,GRN_ARACNE.G12,GRN_ARACNE.G13,GRN_ARACNE.G14,GRN_ARACNE.G15,GRN_ARACNE.G16,GRN_ARACNE.G17,...,GRN_GRENIST.G90,GRN_GRENIST.G91,GRN_GRENIST.G92,GRN_GRENIST.G93,GRN_GRENIST.G94,GRN_GRENIST.G95,GRN_GRENIST.G96,GRN_GRENIST.G97,GRN_GRENIST.G98,GRN_GRENIST.G99
G1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
G10,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
G100,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
G11,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
G12,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
G95,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
G96,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
G97,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
G98,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [181]:
# Function to convert edges to consistent tuple format (e.g., ('G1', 'G10'))
def convert_edges_to_tuples(edge_list):
    edge_tuples = []
    for edge in edge_list:
        gene1, gene2 = edge.split('-')
        # Sort genes to maintain consistency in the edge representation (G1, G10) or (G10, G1)
        edge_tuples.append(tuple(sorted([gene1, gene2])))
    return set(edge_tuples)

# Function to compare predicted edges with known ground truth and generate the metrics
def compare_with_ground_truth(predicted_edges, known_edges_set):
    predicted_set = convert_edges_to_tuples(predicted_edges)
    
    # Calculate true positives and false positives
    true_positives = predicted_set.intersection(known_edges_set)
    false_positives = predicted_set.difference(known_edges_set)
    
    # Metrics
    num_true_positives = len(true_positives)
    num_false_positives = len(false_positives)
    
    return true_positives, false_positives, num_true_positives, num_false_positives

# Example structure to store the results
results = []

# Assume thresholds, iter_thresholds, and common_threshold are defined (as in the previous steps)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        # For each final binarized matrix (e.g., threshold 0.8, iter_threshold 6, common_threshold 2)
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (predicted edges) from the final matrix
        predicted_edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Compare predicted edges with the known ground truth edges
        true_positives, false_positives, num_true_positives, num_false_positives = compare_with_ground_truth(predicted_edges, known_edges_set)
        
        # Create a row for the results table
        results.append({
            'Iterations': num_iterations,
            'Common_Iterations': common_threshold,
            'Iter_Threshold': iter_threshold,
            'Individual_Threshold': threshold,
            'True_Positives': ','.join([f"{tp[0]}-{tp[1]}" for tp in true_positives]),  # Format as G1-G10
            'False_Positives': ','.join([f"{fp[0]}-{fp[1]}" for fp in false_positives]),  # Format as G1-G10
            'No_True_Positives': num_true_positives,
            'No_False_Positives': num_false_positives
        })

# Convert the results to a DataFrame for better visualization
results_df = pd.DataFrame(results)

# Display the results
print(results_df)

# Optionally, save the results to a CSV file
results_df.to_csv('true_false_positives_summary.csv', index=False)


   Iterations  Common_Iterations  Iter_Threshold  Individual_Threshold  \
0          20                 15               8                   0.6   
1          20                 15              10                   0.6   

                                      True_Positives  \
0  G46-G60,G63-G79,G63-G75,G43-G44,G26-G32,G26-G3...   
1  G63-G79,G43-G44,G26-G32,G26-G34,G64-G94,G37-G7...   

                                     False_Positives  No_True_Positives  \
0  G100-G9,G36-G5,G100-G85,G34-G82,G26-G84,G36-G9...                 57   
1  G100-G9,G36-G5,G34-G82,G26-G84,G36-G98,G30-G34...                 44   

   No_False_Positives  
0                 814  
1                 470  


In [183]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

# Function to prepare binary labels (ground truth) and binary predictions
def prepare_binary_labels_and_predictions(final_matrix, known_edges_set, gene_names):
    num_genes = final_matrix.shape[0]
    predictions = []
    labels = []
    
    # Loop through the upper triangle of the matrix (because it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            predicted = final_matrix[i, j]  # This is your prediction (0 or 1)
            edge_tuple = tuple(sorted([gene_names[i], gene_names[j]]))  # Ensure consistent format
            
            predictions.append(predicted)
            # Set ground truth label (1 if edge is in known_edges_set, otherwise 0)
            labels.append(1 if edge_tuple in known_edges_set else 0)
    
    return np.array(predictions), np.array(labels)

# Function to compute AUROC and AUPRC
def compute_roc_and_prc(final_matrix, known_edges_set, gene_names):
    predictions, labels = prepare_binary_labels_and_predictions(final_matrix, known_edges_set, gene_names)
    
    # Calculate AUROC
    auroc = roc_auc_score(labels, predictions)
    
    # Calculate AUPRC
    auprc = average_precision_score(labels, predictions)
    
    return auroc, auprc

# Open a file to save the formatted AUROC and AUPRC results
with open('auroc_auprc_results.txt', 'w') as f:
    # Example: Compute AUROC and AUPRC for each final binarized matrix
    for threshold in thresholds:
        for iter_threshold in iter_thresholds:
            final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
            
            # Compute AUROC and AUPRC
            auroc, auprc = compute_roc_and_prc(final_matrix, known_edges_set, gene_names)
            
            # Format the result strings
            auroc_line = f"AUROC for threshold {threshold}, iter_threshold {iter_threshold}: {auroc}\n"
            auprc_line = f"AUPRC for threshold {threshold}, iter_threshold {iter_threshold}: {auprc}\n"
            
            # Write the formatted results to the file
            f.write(auroc_line)
            f.write(auprc_line)

# Optionally, print a message when the file is saved
print("AUROC and AUPRC results have been saved to 'auroc_auprc_results.txt'.")


AUROC and AUPRC results have been saved to 'auroc_auprc_results.txt'.


In [185]:
predicted_edges

['G1-G25',
 'G1-G29',
 'G1-G37',
 'G1-G48',
 'G1-G61',
 'G1-G68',
 'G1-G69',
 'G1-G72',
 'G1-G73',
 'G1-G74',
 'G1-G77',
 'G10-G12',
 'G10-G14',
 'G10-G16',
 'G10-G17',
 'G10-G39',
 'G10-G43',
 'G10-G44',
 'G10-G45',
 'G10-G63',
 'G10-G64',
 'G10-G79',
 'G10-G9',
 'G10-G91',
 'G10-G92',
 'G10-G93',
 'G100-G39',
 'G100-G9',
 'G11-G15',
 'G11-G17',
 'G11-G26',
 'G11-G27',
 'G11-G5',
 'G11-G6',
 'G11-G98',
 'G11-G99',
 'G12-G5',
 'G12-G86',
 'G12-G98',
 'G14-G16',
 'G14-G17',
 'G14-G38',
 'G14-G43',
 'G14-G44',
 'G14-G63',
 'G14-G91',
 'G14-G92',
 'G15-G17',
 'G15-G21',
 'G15-G26',
 'G15-G27',
 'G15-G30',
 'G15-G34',
 'G15-G5',
 'G15-G54',
 'G15-G6',
 'G15-G81',
 'G15-G82',
 'G15-G98',
 'G15-G99',
 'G16-G17',
 'G16-G52',
 'G16-G56',
 'G17-G38',
 'G17-G43',
 'G17-G56',
 'G17-G91',
 'G17-G92',
 'G18-G22',
 'G18-G37',
 'G18-G46',
 'G18-G47',
 'G18-G48',
 'G18-G52',
 'G18-G53',
 'G18-G56',
 'G18-G57',
 'G18-G58',
 'G18-G60',
 'G19-G20',
 'G19-G28',
 'G19-G38',
 'G19-G44',
 'G19-G45',
 'G19-G5

In [187]:
results_df

,Iterations,Common_Iterations,Iter_Threshold,Individual_Threshold,True_Positives,False_Positives,No_True_Positives,No_False_Positives
0,20,15,8,0.6,"G46-G60,G63-G79,G63-G75,G43-G44,G26-G32,G26-G3...","G100-G9,G36-G5,G100-G85,G34-G82,G26-G84,G36-G9...",57,814
1,20,15,10,0.6,"G63-G79,G43-G44,G26-G32,G26-G34,G64-G94,G37-G7...","G100-G9,G36-G5,G34-G82,G26-G84,G36-G98,G30-G34...",44,470
